# 📊 Dashboard Evaluasi Multi-Threading
## Sistem Evaluasi Optimalisasi Multi-Threading Spesifik

Notebook ini memvisualisasikan distribusi beban kerja per-logical-thread CPU
untuk mendeteksi pola **bottleneck** dan mengevaluasi kualitas **multi-threading**
pada aplikasi web.

### Visualisasi yang Tersedia:
1. **Plot Time-Series** — Overlay beban seluruh thread + arsiran anomali
2. **Boxplot per Fase** — Distribusi beban thread per Testing Phase
3. **Heatmap Crosstab** — Persentase klaster K-Means per fase
4. **Tabel Ringkasan** — Statistik agregat per fase

In [6]:
# ==============================================================================
# SEL 1 · IMPORT LIBRARY & KONFIGURASI
# ==============================================================================

import os
import sys
import warnings

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# Konfigurasi estetika plot global
plt.rcParams.update({
    'figure.figsize': (16, 7),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'font.family': 'sans-serif',
})

sns.set_style('whitegrid')

print('✅ Library berhasil dimuat.')
print(f'   Python  : {sys.version.split()[0]}')
print(f'   Pandas  : {pd.__version__}')
print(f'   Seaborn : {sns.__version__}')

✅ Library berhasil dimuat.
   Python  : 3.13.12
   Pandas  : 2.2.3
   Seaborn : 0.13.2


In [ ]:
# ==============================================================================
# SEL 2 · MUAT DATASET
# ==============================================================================

# Path relatif dari folder notebooks/ ke data/
PATH_ML = os.path.join('..', 'data', 'dataset_dengan_label_ml.csv')
PATH_BERSIH = os.path.join('..', 'data', 'dataset_bersih.csv')

# Prioritas: dataset dengan label ML > dataset bersih
if os.path.isfile(PATH_ML):
    DATASET_PATH = PATH_ML
    print(f'✅ Menggunakan dataset ML: {DATASET_PATH}')
elif os.path.isfile(PATH_BERSIH):
    DATASET_PATH = PATH_BERSIH
    print(f'⚠️  dataset_dengan_label_ml.csv tidak ditemukan.')
    print(f'   Menggunakan fallback: {DATASET_PATH}')
    print(f'   → Jalankan 3_ml_analysis.py untuk visualisasi lengkap.')
else:
    raise FileNotFoundError(
        'Tidak ada dataset ditemukan di folder data/.\n'
        'Jalankan 1_thread_logger.py, 2_data_preprocessing.py, '
        'dan 3_ml_analysis.py terlebih dahulu.'
    )

df = pd.read_csv(DATASET_PATH, encoding='utf-8')

# Konversi Timestamp
if 'Timestamp' in df.columns:
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

# Identifikasi kolom thread secara dinamis
kolom_thread = sorted(
    [c for c in df.columns if c.startswith('Thread_') and c.split('_')[1].isdigit()],
    key=lambda x: int(x.split('_')[1])
)

if not kolom_thread:
    raise ValueError('Tidak ditemukan kolom Thread_X dalam dataset.')

# Tampilkan info dataset
print(f'\n📝 Dataset dimuat: {df.shape[0]} baris × {df.shape[1]} kolom')
print(f'   Thread terdeteksi: {len(kolom_thread)} ({kolom_thread[0]} s/d {kolom_thread[-1]})')

if 'Testing_Phase' in df.columns:
    print(f'   Fase pengujian: {df["Testing_Phase"].unique().tolist()}')

punya_label_ml = 'Anomali_IF' in df.columns and 'KMeans_Label' in df.columns
print(f'   Label ML tersedia: {"Ya" if punya_label_ml else "Tidak"}')

df.head()

✅ Menggunakan dataset ML: ..\data\dataset_dengan_label_ml.csv


ValueError: invalid literal for int() with base 10: 'Imbalance'

---
## 📈 Plot 1: Time-Series Beban Thread (dengan Highlight Anomali)

Line chart overlay beban **Thread_0 s/d Thread_N** sepanjang waktu.
- Setiap thread ditampilkan dengan warna berbeda
- Area **arsiran merah** = rentang waktu terdeteksi **Anomali** (Isolation Forest)
- Garis putus-putus vertikal = batas perubahan fase pengujian

In [ ]:
# ==============================================================================
# SEL 3 · PLOT TIME-SERIES + HIGHLIGHT ANOMALI
# ==============================================================================

# Palet warna yang mudah dibedakan
n_warna = max(len(kolom_thread), 10)
palet_warna = plt.cm.tab10(np.linspace(0, 1, n_warna))

fig, ax = plt.subplots(figsize=(18, 8))

# ---- Plot setiap thread ----
for i, col in enumerate(kolom_thread):
    ax.plot(
        df.index, df[col],
        label=col,
        color=palet_warna[i % len(palet_warna)],
        linewidth=1.2,
        alpha=0.85,
    )

# ---- Arsiran area anomali ----
if 'Anomali_IF' in df.columns:
    mask_anomali = df['Anomali_IF'] == 'Anomali'

    if mask_anomali.any():
        # Kelompokkan indeks anomali yang berurutan menjadi rentang
        idx_list = df.index[mask_anomali].tolist()
        rentang = []
        start = prev = idx_list[0]

        for idx in idx_list[1:]:
            if idx - prev > 1:
                rentang.append((start, prev))
                start = idx
            prev = idx
        rentang.append((start, prev))

        for (r_start, r_end) in rentang:
            ax.axvspan(r_start, r_end, alpha=0.15, color='red', zorder=0)

        # Legend patch untuk anomali
        patch = mpatches.Patch(
            color='red', alpha=0.20,
            label=f'Anomali (Isolation Forest) — {mask_anomali.sum()} titik'
        )
        handles, _ = ax.get_legend_handles_labels()
        handles.append(patch)
        ax.legend(handles=handles, loc='upper right', framealpha=0.9, ncol=2)
    else:
        ax.legend(loc='upper right', framealpha=0.9, ncol=2)
        print('ℹ️  Tidak ada anomali terdeteksi oleh Isolation Forest.')
else:
    ax.legend(loc='upper right', framealpha=0.9, ncol=2)
    print('ℹ️  Kolom Anomali_IF tidak tersedia. Jalankan 3_ml_analysis.py.')

# ---- Batas antar fase pengujian ----
if 'Testing_Phase' in df.columns:
    perubahan_fase = df['Testing_Phase'].ne(df['Testing_Phase'].shift())
    for idx in df.index[perubahan_fase]:
        if idx > 0:
            ax.axvline(x=idx, color='gray', linestyle='--', alpha=0.5, linewidth=1)
            ax.text(
                idx, 102, df.loc[idx, 'Testing_Phase'],
                rotation=45, fontsize=8, ha='left', va='bottom',
                color='gray', fontstyle='italic'
            )

ax.set_title(
    'Distribusi Beban per Logical Thread Sepanjang Waktu\n'
    '(Arsiran Merah = Anomali Terdeteksi)',
    fontsize=14, fontweight='bold'
)
ax.set_xlabel('Indeks Waktu (detik ke-N)')
ax.set_ylabel('Beban CPU (%)')
ax.set_ylim(-2, 108)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 📦 Plot 2: Boxplot Distribusi Beban Thread per Fase Pengujian

Boxplot menunjukkan sebaran beban (median, Q1, Q3, outlier) masing-masing thread,
dikelompokkan per **Testing_Phase**.

- Jika distribusi **merata** → boxplot serupa antar thread → multi-threading optimal
- Jika satu thread **jauh lebih tinggi** → indikasi **bottleneck**
- Titik merah (◆) = **rata-rata** beban thread

In [ ]:
# ==============================================================================
# SEL 4 · BOXPLOT PER FASE PENGUJIAN
# ==============================================================================

if 'Testing_Phase' not in df.columns:
    print('⚠️  Kolom Testing_Phase tidak tersedia. Boxplot tidak dapat dibuat.')
else:
    fase_list = sorted(df['Testing_Phase'].unique())
    n_fase = len(fase_list)

    fig, axes = plt.subplots(
        nrows=n_fase, ncols=1,
        figsize=(16, 5 * max(n_fase, 1)),
        squeeze=False,
    )

    palet = sns.color_palette('husl', n_colors=len(kolom_thread))

    for i, fase in enumerate(fase_list):
        ax = axes[i, 0]
        subset = df[df['Testing_Phase'] == fase][kolom_thread]

        # Melt ke format long untuk seaborn
        df_melt = subset.melt(var_name='Thread', value_name='Beban_CPU_Persen')

        sns.boxplot(
            data=df_melt,
            x='Thread',
            y='Beban_CPU_Persen',
            palette=palet,
            ax=ax,
            width=0.6,
            fliersize=3,
            linewidth=1.2,
        )

        ax.set_title(
            f'Distribusi Beban Thread \u2014 Fase: \"{fase}\" ({len(subset)} data poin)',
            fontsize=13, fontweight='bold'
        )
        ax.set_xlabel('Logical Thread')
        ax.set_ylabel('Beban CPU (%)')
        ax.set_ylim(-2, 105)
        ax.grid(True, axis='y', alpha=0.3)

        # Titik mean (diamond merah)
        means = subset.mean()
        for j, col in enumerate(kolom_thread):
            ax.plot(j, means[col], 'D', color='red', markersize=5, zorder=5)

        # Annotasi Imbalance Score
        if 'Thread_Imbalance_Score' in df.columns:
            imb = df.loc[df['Testing_Phase'] == fase, 'Thread_Imbalance_Score'].mean()
            ax.text(
                0.98, 0.95,
                f'Avg Imbalance Score: {imb:.3f}',
                transform=ax.transAxes,
                ha='right', va='top',
                fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8)
            )

    fig.text(
        0.5, -0.01,
        '◆ Titik merah = rata-rata beban thread  |  Box = Q1\u2013Q3  |  Garis = median',
        ha='center', fontsize=10, fontstyle='italic', color='gray'
    )

    plt.tight_layout()
    plt.show()

---
## 📋 Ringkasan Statistik per Fase Pengujian

In [ ]:
# ==============================================================================
# SEL 5 · TABEL RINGKASAN STATISTIK
# ==============================================================================

# Kolom numerik yang ingin diringkas
kolom_ringkasan = [
    'CPU_Total_Persen', 'Memory_Usage_MB',
    'Max_Thread_Load', 'Thread_Imbalance_Score', 'Thread_Range'
]
kolom_tersedia = [c for c in kolom_ringkasan if c in df.columns]

if 'Testing_Phase' in df.columns and kolom_tersedia:
    tabel = (
        df.groupby('Testing_Phase')[kolom_tersedia]
        .agg(['mean', 'std', 'max'])
        .round(2)
    )
    print('📊 Statistik Agregat per Fase Pengujian:\n')
    display(tabel)
elif not kolom_tersedia:
    print('⚠️  Tidak ada kolom ringkasan numerik yang tersedia.')
else:
    print('⚠️  Kolom Testing_Phase tidak tersedia.')

In [ ]:
# ==============================================================================
# SEL 6 · DISTRIBUSI KLASTER K-MEANS PER FASE (HEATMAP)
# ==============================================================================

if 'KMeans_Label' in df.columns and 'Testing_Phase' in df.columns:
    # Crosstab absolut
    cross_abs = pd.crosstab(
        df['Testing_Phase'],
        df['KMeans_Label'],
        margins=True,
        margins_name='Total'
    )
    print('\n📊 Crosstab: Distribusi Klaster K-Means per Fase Pengujian\n')
    display(cross_abs)

    # Heatmap persentase
    cross_pct = pd.crosstab(
        df['Testing_Phase'],
        df['KMeans_Label'],
        normalize='index'
    ) * 100

    fig, ax = plt.subplots(figsize=(10, max(3, len(cross_pct) * 1.2)))
    sns.heatmap(
        cross_pct,
        annot=True, fmt='.1f', cmap='YlOrRd',
        linewidths=0.5, ax=ax,
        cbar_kws={'label': 'Persentase (%)'},
    )
    ax.set_title('Persentase Klaster per Fase Pengujian (%)',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('Fase Pengujian')
    ax.set_xlabel('Klaster K-Means')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Kolom KMeans_Label tidak tersedia.')
    print('   → Jalankan 3_ml_analysis.py terlebih dahulu.')

---
## ✅ Kesimpulan

Berdasarkan analisis di atas:

| Kondisi | Interpretasi |
|---------|:-------------|
| Semua boxplot distribusi serupa | Aplikasi memanfaatkan multi-threading dengan baik |
| Satu thread mendominasi | Terdapat **bottleneck** yang perlu dioptimalisasi |
| Area arsiran merah (time-series) | Momen **anomali** yang perlu diinvestigasi |
| Heatmap didominasi warna gelap pada "Bottleneck" | Fase tersebut **kritis** |

Gunakan insight ini untuk menentukan apakah arsitektur asinkron/multi-thread
aplikasi web sudah optimal, atau perlu perbaikan pada komponen tertentu.